# 01 - Exploration de Google ISLR (asl-signs)

Objectifs : nombre d'exemples par signeur et par signe, forme d'une sequence, longueurs,
proportion de points manquants par type, main dominante par signeur.
Les resultats sont ecrits dans `/kaggle/working/reports/` (a recuperer dans le depot, dossier `reports/`).

In [ ]:
# Acces au code du projet : soit un dataset Kaggle "lsf-code" (dossier src/ du depot),
# soit un clone GitHub (activer Internet dans les reglages du notebook).
import glob, os, subprocess, sys
CODE_DATASET = "/kaggle/input/lsf-code/src"
REPO = "https://github.com/M-Jel/LSF_project.git"  # A CONFIRMER : nom du depot GitHub
if os.path.exists(CODE_DATASET):
    sys.path.insert(0, CODE_DATASET)
else:
    if not os.path.exists("/kaggle/working/LSF_project"):
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/LSF_project"], check=True)
    sys.path.insert(0, "/kaggle/working/LSF_project/src")

# Localisation automatique des donnees de la competition asl-signs
ROOT = os.path.dirname(sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True))[0])
print("donnees :", ROOT)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lsf.data.islr import load_sequence, verify_layout
from lsf.features.landmarks import OFFSETS

OUT = "/kaggle/working/reports"
os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(f"{ROOT}/train.csv")
print(df.shape); df.head()

## Exemples par signeur et par signe

In [ ]:
per_signer = df.groupby("participant_id").size().sort_values()
per_sign = df.groupby("sign").size().sort_values()
print("signeurs :", df.participant_id.nunique(), "| signes :", df.sign.nunique())
print(per_signer.describe()); print(per_sign.describe())
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
per_signer.plot.bar(ax=ax[0], title="Exemples par signeur")
per_sign.plot.hist(ax=ax[1], bins=30, title="Exemples par signe")
plt.tight_layout(); plt.savefig(f"{OUT}/counts.png", dpi=120)
# Matrice signeur x signe : un signeur a-t-il fait tous les signes ?
cover = (df.groupby(["participant_id", "sign"]).size().unstack(fill_value=0) > 0).sum(1)
print("signes couverts par signeur :", cover.min(), "a", cover.max())

## Disposition des points et forme d'une sequence

In [ ]:
first = f"{ROOT}/{df.path[0]}"
print("disposition :", verify_layout(first))
raw = load_sequence(first)
print("forme (frames, points, xyz) :", raw.shape)
pd.read_parquet(first).head()

## Echantillon : longueurs, points manquants, main dominante

Echantillon aleatoire (seed 0) pour limiter le temps de lecture.

In [ ]:
from multiprocessing import Pool
sample = df.sample(5000, random_state=0).reset_index(drop=True)
SLICES = {"face": slice(0, 468), "left_hand": slice(468, 489), "pose": slice(489, 522), "right_hand": slice(522, 543)}

def stats(path):
    r = load_sequence(f"{ROOT}/{path}")
    absent = {k: float(np.isnan(r[:, s, 0]).all(1).mean()) for k, s in SLICES.items()}
    lh = ~np.isnan(r[:, SLICES["left_hand"], 0]).all(1)
    rh = ~np.isnan(r[:, SLICES["right_hand"], 0]).all(1)
    return {"frames": len(r), **{f"absent_{k}": v for k, v in absent.items()},
            "no_hand": float((~lh & ~rh).mean()), "left_frames": int(lh.sum()), "right_frames": int(rh.sum()),
            "z_min": float(np.nanmin(r[..., 2])), "z_max": float(np.nanmax(r[..., 2]))}

with Pool(4) as p:
    st = pd.DataFrame(p.map(stats, sample.path, chunksize=50))
st = pd.concat([sample, st], axis=1)
st.to_csv(f"{OUT}/sample_stats.csv", index=False)
st.filter(regex="frames|absent|no_hand").describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
st.frames.clip(upper=200).plot.hist(bins=50, ax=ax[0], title="Frames par sequence (coupe a 200)")
st.filter(like="absent_").mean().plot.bar(ax=ax[1], title="Proportion de frames ou le type est absent")
plt.tight_layout(); plt.savefig(f"{OUT}/lengths_missing.png", dpi=120)
for q in (0.5, 0.9, 0.95, 0.99):
    print(f"quantile {q} des longueurs : {st.frames.quantile(q):.0f} frames")
print("part des sequences > 64 frames :", (st.frames > 64).mean().round(3))

## Main dominante par signeur (biais a documenter)

In [ ]:
st["dominant"] = np.where(st.left_frames > st.right_frames, "left_hand", "right_hand")
dom = pd.crosstab(st.participant_id, st.dominant, normalize="index").round(2)
dom.to_csv(f"{OUT}/dominant_hand_by_signer.csv"); dom

In [ ]:
summary = {
    "n_sequences": len(df), "n_signers": int(df.participant_id.nunique()), "n_signs": int(df.sign.nunique()),
    "per_signer_min_max": [int(per_signer.min()), int(per_signer.max())],
    "frames_quantiles": st.frames.quantile([.5, .9, .95, .99]).to_dict(),
    "absent_mean": st.filter(like="absent_").mean().round(4).to_dict(),
    "no_hand_frames_mean": round(float(st.no_hand.mean()), 4),
}
json.dump(summary, open(f"{OUT}/exploration_summary.json", "w"), indent=2)
summary